# BMA — Kaggle Setup, Phase 0 Smoke Test, and Phase 1 Corpus EDA

This notebook is ready to run in Kaggle with **Run All**. It clones or updates the public repository, installs the local `doc_agent` package, verifies the environment and GPU, discovers attached PDF inputs, loads the project configuration, and writes a smoke-test artifact.

Before running, attach the two source PDFs as a **private Kaggle dataset**. The notebook discovers PDFs anywhere under `/kaggle/input`, so the Kaggle dataset slug does not need to be hard-coded.

In [ ]:
# Configuration — normally no edits are required.
from pathlib import Path

REPO_URL = "https://github.com/FAHIM-ISHTIAK/doc-agent-20.git"
REPO_BRANCH = "fahim"  # Phase branch; change to main after the PR is merged.
REPO_DIR = Path("/kaggle/working/doc-agent-20")
KAGGLE_INPUT = Path("/kaggle/input")
ARTIFACT_DIR = Path("/kaggle/working/artifacts")

print("Repository:", REPO_URL)
print("Branch:", REPO_BRANCH)
print("Working copy:", REPO_DIR)
print("Kaggle inputs:", KAGGLE_INPUT)


## 1. Inspect the Kaggle runtime

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Current directory:", Path.cwd())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for gpu_index in range(torch.cuda.device_count()):
            print(f"GPU {gpu_index}:", torch.cuda.get_device_name(gpu_index))
    else:
        print("WARNING: No GPU detected. In Kaggle, open Settings -> Accelerator and select a GPU.")
except Exception as exc:
    print("PyTorch inspection failed:", repr(exc))


## 2. Clone or update the repository

The cell is safe to run again. If a clean Git checkout already exists, it updates `main` with a fast-forward pull. It does not delete or overwrite an unrelated directory.

In [ ]:
def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)))
    completed = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository. Choose a different REPO_DIR.")
    print("Existing repository found; updating it.")
    run_command(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_command(["git", "switch", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    print("Cloning repository.")
    run_command(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

commit_hash = run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
status = run_command(["git", "status", "--short"], cwd=REPO_DIR).stdout.strip()
print("Tested commit:", commit_hash)
print("Working tree:", "clean" if not status else status)


## 3. Install the repository package

This smoke test installs the local package without resolving the full ML dependency stack. Phase 2 will finalize and pin the complete Kaggle dependency environment.

In [ ]:
run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "pymupdf>=1.24,<1.27",
        "pillow>=10.3,<12.0",
        "pyyaml>=6.0,<7.0",
        "pydantic>=2.7,<3.0",
    ]
)
run_command(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"]
)

# Add src explicitly as a robust notebook-session fallback after editable installation.
src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

os.chdir(REPO_DIR)
print("Current directory:", Path.cwd())
print("Source directory on sys.path:", str(src_dir) in sys.path)


## 4. Import `doc_agent` and load its fixed contracts

In [ ]:
import importlib.util

package_spec = importlib.util.find_spec("doc_agent")
print("doc_agent module specification:", package_spec)
if package_spec is None:
    raise ModuleNotFoundError("doc_agent is still unavailable after editable installation")

import doc_agent
from doc_agent import config
from doc_agent.contracts import Chunk, Page, Region

cfg = config.load(REPO_DIR / "configs" / "config.yaml")
task_cfg = config.load_task(REPO_DIR / "configs" / "task.yaml")

print("doc_agent imported from:", Path(doc_agent.__file__).resolve())
print("Configuration loaded successfully:")
print(cfg)
print("Task configuration loaded successfully:")
print(task_cfg)
print("Contracts:", Page.__name__, Region.__name__, Chunk.__name__)
print("IMPORT SMOKE TEST: PASS")


## 5. Discover attached corpus PDFs

The PDFs may live under any private Kaggle input dataset. This cell searches all of `/kaggle/input`.

In [ ]:
if not KAGGLE_INPUT.exists():
    raise FileNotFoundError("/kaggle/input does not exist. This notebook is intended to run in Kaggle.")

pdf_files = sorted(KAGGLE_INPUT.rglob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s) under {KAGGLE_INPUT}:")
for pdf_path in pdf_files:
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    print(f"- {pdf_path} ({size_mb:.2f} MB)")

if len(pdf_files) < 2:
    raise FileNotFoundError(
        "Expected the two BMA source PDFs. Attach the private corpus dataset to this Kaggle notebook and run again."
    )

print("CORPUS DISCOVERY: PASS")


## 6. Validate the fixed data contracts

In [ ]:
sample_page = Page(
    id="smoke_page_0001",
    image_path="/kaggle/working/example.png",
    doc_id="smoke_document",
)
sample_region = Region(
    page_id=sample_page.id,
    bbox=(0, 0, 100, 100),
    kind="text",
)
sample_chunk = Chunk(
    id="smoke_chunk_0001",
    doc_id=sample_page.doc_id,
    text="বাংলা ব্যাকরণ",
    page_ids=[sample_page.id],
)

print(sample_page.model_dump())
print(sample_region.model_dump())
print(sample_chunk.model_dump())
assert sample_chunk.text == "বাংলা ব্যাকরণ"
assert sample_chunk.page_ids == [sample_page.id]
print("CONTRACT SMOKE TEST: PASS")


## 7. Verify writable artifact storage

In [ ]:
import json
from datetime import datetime, timezone

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
smoke_artifact = ARTIFACT_DIR / "phase0_smoke_test.json"
smoke_payload = {
    "status": "pass",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_URL,
    "branch": REPO_BRANCH,
    "commit": commit_hash,
    "python": sys.version,
    "pdf_count": len(pdf_files),
    "pdf_files": [str(path) for path in pdf_files],
}
smoke_artifact.write_text(
    json.dumps(smoke_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(smoke_artifact.read_text(encoding="utf-8"))
print("Artifact written to:", smoke_artifact)
print("WRITABLE STORAGE TEST: PASS")


## Phase 0 result

If every code cell above finishes successfully, the Kaggle portion of Phase 0 is complete:

- the public repository can be cloned or updated;
- the exact tested commit is recorded;
- `doc_agent` imports from the repository's `src/` layout;
- the fixed `Page`, `Region`, and `Chunk` contracts work;
- the private corpus PDFs are visible without hard-coded dataset slugs;
- the selected GPU/runtime is reported;
- output can be written to `/kaggle/working/artifacts`.

The remaining cells perform Phase 1 against the exact Team 20 corpus.

# Phase 1 — Corpus identity, integrity, and EDA

These cells validate the exact Team 20 files, calculate reproducible metadata, inspect real page-image properties, display representative scans, declare the document-level split, and export Phase 1 artifacts. Embedded PDF text is not used as the system OCR output.

## 8. Identify and validate the exact corpus files

In [ ]:
import hashlib

import fitz

EXPECTED_CORPUS = {
    "bhasha_prakash_1942": {
        "filename": "bhasha_prakash_1942.pdf",
        "expected_pages": 565,
        "expected_bytes": 26_715_873,
        "expected_sha256": "2e6a6e08e942d91e0986e2282ddeec9e6b4882ac6f4b89e083bf9bab6e72d39b",
        "split": "train",
        "year": 1942,
    },
    "nctb_bangla_grammar_2026": {
        "filename": "nctb_bangla_grammar.pdf",
        "expected_pages": 218,
        "expected_bytes": 143_145_319,
        "expected_sha256": "3cb900fb1ce51bb91e13e41d43c51d98260a29ffffbed5632b0adee130640086",
        "split": "test",
        "year": 2026,
    },
}


def file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


pdf_by_name = {path.name: path for path in pdf_files}
corpus_records = []
corpus_paths = {}

for doc_id, expected in EXPECTED_CORPUS.items():
    filename = expected["filename"]
    if filename not in pdf_by_name:
        raise FileNotFoundError(f"Missing declared corpus file: {filename}")

    source_path = pdf_by_name[filename]
    size_bytes = source_path.stat().st_size
    digest = file_sha256(source_path)
    with fitz.open(source_path) as pdf:
        pages = len(pdf)

    assert size_bytes == expected["expected_bytes"], (filename, size_bytes)
    assert digest == expected["expected_sha256"], (filename, digest)
    assert pages == expected["expected_pages"], (filename, pages)

    corpus_paths[doc_id] = source_path
    corpus_records.append(
        {
            "doc_id": doc_id,
            "filename": filename,
            "path": str(source_path),
            "year": expected["year"],
            "split": expected["split"],
            "pages": pages,
            "bytes": size_bytes,
            "size_mib": round(size_bytes / (1024 * 1024), 2),
            "sha256": digest,
        }
    )

for record in corpus_records:
    print(json.dumps(record, ensure_ascii=False, indent=2))

total_pages = sum(record["pages"] for record in corpus_records)
total_bytes = sum(record["bytes"] for record in corpus_records)
assert total_pages == 783
print(f"Combined corpus: {total_pages} pages, {total_bytes:,} bytes ({total_bytes / (1024**2):.2f} MiB)")
print("CORPUS IDENTITY AND INTEGRITY: PASS")


## 9. Measure page geometry and rotation

This reads PDF page metadata only. It does not use the embedded text layer as OCR.

In [ ]:
from collections import Counter

geometry_records = []
for doc_id, source_path in corpus_paths.items():
    sizes = Counter()
    rotations = Counter()
    portrait = 0
    landscape = 0

    with fitz.open(source_path) as pdf:
        for page in pdf:
            width = round(page.rect.width, 2)
            height = round(page.rect.height, 2)
            sizes[(width, height)] += 1
            rotations[page.rotation] += 1
            if height >= width:
                portrait += 1
            else:
                landscape += 1

    record = {
        "doc_id": doc_id,
        "portrait_pages": portrait,
        "landscape_pages": landscape,
        "rotations": dict(rotations),
        "most_common_page_sizes_points": [
            {"width": size[0], "height": size[1], "pages": count}
            for size, count in sizes.most_common(5)
        ],
    }
    geometry_records.append(record)
    print(json.dumps(record, ensure_ascii=False, indent=2))

print("PAGE GEOMETRY EDA: PASS")


## 10. Display representative real pages

The selected pages cover both documents and include the NCTB rule/list area previously discussed in A1. Page numbers below are one-based PDF indices.

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

SAMPLE_PAGES = {
    "bhasha_prakash_1942": [10, 150, 350],
    "nctb_bangla_grammar_2026": [10, 42, 100],
}

figure, axes = plt.subplots(2, 3, figsize=(16, 18))
sample_scan_stats = []

for row_index, (doc_id, page_numbers) in enumerate(SAMPLE_PAGES.items()):
    with fitz.open(corpus_paths[doc_id]) as pdf:
        for column_index, pdf_page_number in enumerate(page_numbers):
            page = pdf.load_page(pdf_page_number - 1)
            pixmap = page.get_pixmap(dpi=120, colorspace=fitz.csGRAY, alpha=False)
            image = Image.open(BytesIO(pixmap.tobytes("png"))).convert("L")
            pixels = np.asarray(image, dtype=np.uint8)
            sample_scan_stats.append(
                {
                    "doc_id": doc_id,
                    "pdf_page": pdf_page_number,
                    "width_px_at_120dpi": image.width,
                    "height_px_at_120dpi": image.height,
                    "mean_gray": round(float(pixels.mean()), 2),
                    "gray_std": round(float(pixels.std()), 2),
                    "dark_pixel_fraction_lt_128": round(float((pixels < 128).mean()), 4),
                }
            )
            axis = axes[row_index, column_index]
            axis.imshow(image, cmap="gray")
            axis.set_title(f"{doc_id} — PDF page {pdf_page_number}")
            axis.axis("off")

plt.tight_layout()
plt.show()

for record in sample_scan_stats:
    print(record)
print("REAL PAGE SAMPLE EDA: PASS")


## 11. Declare the document-level split and corpus revision

In [ ]:
split_by_document = {record["doc_id"]: record["split"] for record in corpus_records}
assert split_by_document == {
    "bhasha_prakash_1942": "train",
    "nctb_bangla_grammar_2026": "test",
}
assert len(split_by_document) == len(set(split_by_document))

print("Document-level split:", split_by_document)
print("Validation: one whole-chapter block will be selected from the 1942 train volume after layout inspection.")
print("Limitation: validation shares the train document's era, author, and typography.")
print("Corpus revision: A1 named the 2019 NCTB edition; A2 uses the selected 2026 218-page edition.")
print("The revision must be disclosed in A2 and confirmed with the instructor.")
print("DOCUMENT SPLIT DECLARATION: PASS")


## 12. Export the Phase 1 inventory and summary

In [ ]:
phase1_inventory = {
    "status": "pass",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_URL,
    "branch": REPO_BRANCH,
    "commit": commit_hash,
    "documents": corpus_records,
    "total_pages": total_pages,
    "total_bytes": total_bytes,
    "total_size_mib": round(total_bytes / (1024 * 1024), 2),
    "word_count": None,
    "word_count_status": "pending measured OCR in Phase 6",
    "geometry": geometry_records,
    "sample_scan_statistics": sample_scan_stats,
    "split_by_document": split_by_document,
    "corpus_revision_from_a1": "2019 NCTB edition replaced by selected 2026 218-page edition",
}

inventory_path = ARTIFACT_DIR / "phase1_corpus_inventory.json"
inventory_path.write_text(
    json.dumps(phase1_inventory, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary_lines = [
    "# Phase 1 corpus summary",
    "",
    f"- Tested repository commit: `{commit_hash}`",
    f"- Documents: {len(corpus_records)}",
    f"- Raw pages: {total_pages}",
    f"- Source size: {total_bytes:,} bytes ({total_bytes / (1024**2):.2f} MiB)",
    "- Usable OCR words: pending Phase 6",
    "- Split: 1942 volume train; 2026 NCTB volume test; whole-chapter validation block pending",
    "- Corpus revision: A1's 2019 NCTB edition was replaced by the selected 2026 edition",
]
summary_path = ARTIFACT_DIR / "phase1_corpus_summary.md"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print(inventory_path.read_text(encoding="utf-8"))
print("Exported:", inventory_path)
print("Exported:", summary_path)
print("PHASE 1 CORPUS INVENTORY: PASS")


## Phase 1 result and remaining gate

Phase 1's reproducible corpus identity/EDA work is complete when all cells show PASS and a Kaggle version is saved with outputs. Download the executed notebook and the two files under `/kaggle/working/artifacts`.

Two values intentionally remain open:

1. the exact usable word count, which must come from measured OCR in Phase 6 rather than a PDF text layer;
2. the exact validation chapter block, which is selected after layout inspection while remaining entirely inside the 1942 train document.

The team must also disclose/confirm the change from A1's 2019 NCTB edition to the 2026 edition.